In [ ]:
from pathlib import Path
from typing import Literal, Dict

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency
from category_encoders import TargetEncoder

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.model_selection import train_test_split

d:\Python\classic_ml\binary_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv("../dataframes/prepared/loan_feature_extracted.csv")

In [ ]:
number_cols = df.select_dtypes("number").columns
cat_cols = df.select_dtypes(include=["object", "string", "category"]).columns

In [ ]:
display(df["Employment_Status"].value_counts())
display(df["Employer_Category"].value_counts())
display(df["Loan_Purpose"].value_counts())

In [ ]:
check_emp_stat_means = df.groupby("Employment_Status")["Loan_Approved"].mean()
check_emp_cat_means = df.groupby("Employer_Category")["Loan_Approved"].mean()
check_loan_purp_means = df.groupby("Loan_Purpose")["Loan_Approved"].mean()

display(check_emp_stat_means)
display(check_emp_cat_means)
display(check_loan_purp_means)

In [ ]:
for c in cat_cols:
    print("\n", c)
    print(df[c].unique())

In [ ]:
cols_for_ohe = ["Marital_Status", "Education_Level", "Gender", "Loan_Purpose"]

In [ ]:
ohe_scaler = OneHotEncoder(sparse_output=False)
ohe_scaler.fit(df[cols_for_ohe])

encoded_ohe = ohe_scaler.transform(df[cols_for_ohe])

encoded_cols = ohe_scaler.get_feature_names_out(cols_for_ohe)

df = pd.concat(
    [
        df.drop(columns=cols_for_ohe),
        pd.DataFrame(encoded_ohe, columns=encoded_cols, index=df.index),
    ],
    axis=1,
)

In [ ]:
ordinal_cols = ["Property_Area"]
ordinal_cat = ["Rural", "Semiurban", "Urban"]

In [ ]:
oe_scaler = OrdinalEncoder(categories=[ordinal_cat])
oe_scaler.fit(df[ordinal_cols])

df[ordinal_cols] = oe_scaler.transform(df[ordinal_cols])

In [ ]:
preprocessed_cols = df.select_dtypes("number").columns

In [ ]:
plt.figure(figsize=(18, 18))
sns.heatmap(df[preprocessed_cols].corr(), annot=True, fmt=".1f", linewidths=0.5)
plt.show()

In [ ]:
df[number_cols].corr()["Loan_Approved"].sort_values()

In [ ]:
sns.kdeplot(df[df["Loan_Approved"] == 0]["Credit_Score"], label="0")
sns.kdeplot(df[df["Loan_Approved"] == 1]["Credit_Score"], label="1")
plt.legend()
plt.show()

In [ ]:
def check_chi2(data, cols):
    for col in cols:
        contingency_table = pd.crosstab(data[col], data["Loan_Approved"])
        chi2, p, dof, ex = chi2_contingency(contingency_table)

        print(f"Column: {col}")
        print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}")
        if p < 0.05:
            print("Related to the target\n")
        else:
            print("Is not related to the target\n")

In [ ]:
check_chi2(df, ["Employment_Status", "Employer_Category"])

In [ ]:
df.drop(["Employer_Category"], inplace=True, axis=1)

In [ ]:
def dataset_splitter(
    X, y, model_type: Literal["tree", "linear"], test_size: float = 0.3
):
    if model_type == "linear":
        X.drop(
            ["Applicant_Income", "Coapplicant_Income", "risk_score"],
            inplace=True,
            axis=1,
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    return X_train, X_test, y_train, y_test

In [ ]:
X = df.drop(columns=["Loan_Approved"])
y = df["Loan_Approved"]

In [ ]:
X_train, X_test, y_train, y_test = dataset_splitter(X, y, model_type="linear")

In [ ]:
def data_retention(data: Dict[str, pd.DataFrame], prefix: str, path: str):
    saving_path = Path(path)
    if not saving_path.exists():
        saving_path.mkdir(parents=True, exist_ok=True)

    for k, d in data.items():
        d.to_csv(f"{path}/{k}_{prefix}.csv")

### BELOW SOLUTIONS FOR LINEAR MODELS ONLY

In [ ]:
len(X_train.select_dtypes(include="object").columns)

In [ ]:
linear_num_cols = X_train.select_dtypes("number").columns

In [ ]:
def get_outliers_iqr(df, cols):
    limits = {}

    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        upper_outliers = Q3 + 1.5 * IQR
        lower_outliers = Q1 - 1.5 * IQR

        limits[col] = (upper_outliers, lower_outliers)
        outliers = df[col][(df[col] < lower_outliers) | (df[col] > upper_outliers)]

        print(f"\nCOLUMN: '{col}'")
        print(f"Upper limit of outliers: {upper_outliers}")
        print(f"Lower limit of outliers: {lower_outliers}")
        print(f"Amount of outliers: {len(outliers)}")

    return limits

In [ ]:
def boxplot_vizualizer(df, cols: int = 2):
    rows = round(len(df.columns) / cols)

    fig, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(8, 16))

    for i, col in enumerate(df.columns):
        r = i // cols
        c = i % cols

        sns.boxplot(y=df[col], ax=ax[r, c])
        ax[r, c].set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
rb_scaler = RobustScaler()
rb_scaler.fit(X_train[linear_num_cols])

X_train[linear_num_cols] = rb_scaler.transform(X_train[linear_num_cols])
X_test[linear_num_cols] = rb_scaler.transform(X_test[linear_num_cols])

In [ ]:
limits = get_outliers_iqr(X_train, cols=linear_num_cols)

for col, (lower, upper) in limits.items():
    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

In [ ]:
boxplot_vizualizer(X_train[linear_num_cols])

### THIS IS THE END OF SOLUTION FOR LINEAR MODELS

### COOMON STAGE

In [ ]:
target_cols = ["Employment_Status"]

In [ ]:
target_enc = TargetEncoder(smoothing=5)
target_enc.fit(X_train[target_cols], y_train)

X_train[target_cols] = target_enc.transform(X_train[target_cols])
X_test[target_cols] = target_enc.transform(X_test[target_cols])

In [ ]:
data_to_save = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
}

In [ ]:
data_retention(
    data=data_to_save, prefix="with_target_enc", path="../dataframes/prepared/for_linear/with_target_enc"
)